# Stage 3: GRPO Training - Creative Code Reward Reinforcement

**Cipher Code Kraken Training Pipeline - Stage 3 of 4**

GRPO (Group Relative Policy Optimization) generates multiple completions per prompt and uses a reward function to reinforce creative code quality. Unlike SimPO which learns from fixed preference pairs, GRPO actively explores the generation space and optimizes toward higher-reward outputs.

**Input:** `cipher-simpo-merged` (from Stage 2)
**Output:** `cipher-grpo-merged` (input for Stage 4: KTO)

**Key features:**
- Enhanced reward function with anti-hacking measures (Pitfall 3 mitigation)
- Import validation, repetition detection, code structure checks
- Reward hacking monitoring via W&B
- Unsloth GRPO with TRL fallback (Open Question 1)

**Runtime:** Colab Pro+ A100 40GB | ~4-8 hours | ~60 compute units

## Cell 1: Environment Setup
Install Unsloth and W&B. Log in to W&B for experiment tracking.

In [ ]:
!pip install unsloth
!pip install wandb
import wandb
wandb.login()

## Cell 2: Mount Drive and Copy Data
Mount Google Drive and copy the SimPO-merged model, GRPO prompts dataset, and scripts to the Colab local filesystem for faster I/O.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy SimPO-merged model (output from Stage 2)
!cp -r /content/drive/MyDrive/cipher-models/cipher-simpo-merged ./

# Copy GRPO prompts dataset
!mkdir -p data/prompts
!cp /content/drive/MyDrive/cipher-data/grpo_prompts.jsonl ./data/prompts/

# Copy scripts (slop_detector for reward function)
!cp -r /content/drive/MyDrive/cipher-scripts/scripts ./

# Copy configs
!mkdir -p configs
!cp /content/drive/MyDrive/cipher-scripts/configs/grpo_config.py ./configs/
!cp /content/drive/MyDrive/cipher-scripts/configs/__init__.py ./configs/

print("Data mounted and copied successfully")
!ls -la cipher-simpo-merged/ | head -5
!wc -l data/prompts/grpo_prompts.jsonl

## Cell 3: Load SimPO-Merged Model
Load the model from Stage 2 output with QLoRA 4-bit quantization. Monitor VRAM usage.

In [ ]:
from unsloth import FastLanguageModel
import torch
import sys
sys.path.append('/content')
from configs.grpo_config import *

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_ID,
    load_in_4bit=LOAD_IN_4BIT,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
)
print(f"SimPO-merged model loaded. VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 4: Apply LoRA Adapter
Apply a fresh LoRA adapter for GRPO training. Same rank/alpha as previous stages for consistency.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODULES,
    use_gradient_checkpointing=USE_GRADIENT_CHECKPOINTING,
    random_state=RANDOM_STATE,
)
model.print_trainable_parameters()
print(f"Post-LoRA VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

## Cell 5: Define Enhanced Reward Function
Import the base reward from slop_detector, then wrap it with anti-hacking measures.

**Pitfall 3 Mitigation:** The enhanced reward checks code STRUCTURE, not just keywords:
- Import validation: are imported modules actually used in the code?
- Variable definition checks: are variables defined before use?
- Repetition detection: penalizes >50% repeated lines (reward gaming signal)
- Code block validation: ensures response contains actual code blocks

This prevents the model from gaming rewards by keyword-stuffing Three.js/GSAP terms
without writing coherent code.

In [ ]:
# Import base reward from slop_detector, then enhance with anti-hacking measures
try:
    from scripts.slop_detector import creative_code_reward as base_reward
    print("slop_detector loaded successfully")
except ImportError:
    print("WARNING: slop_detector not found. Using inline fallback.")
    def base_reward(completions, **kwargs):
        """Fallback reward: basic keyword detection."""
        rewards = []
        for c in completions:
            score = 0.0
            if 'THREE.' in c or 'three' in c: score += 2.0
            if 'gsap.' in c or 'ScrollTrigger' in c: score += 2.0
            if 'Lenis' in c: score += 1.5
            if 'bg-gradient-to' in c: score -= 2.0
            if 'animate-bounce' in c: score -= 2.0
            rewards.append(score)
        return rewards

import re

def enhanced_creative_reward(completions, **kwargs):
    """Enhanced reward with anti-hacking measures (Pitfall 3 mitigation).
    
    Wraps the base creative_code_reward with structural checks that
    prevent the model from gaming rewards by keyword-stuffing.
    """
    base_rewards = base_reward(completions)
    enhanced_rewards = []
    
    for i, (completion, base_score) in enumerate(zip(completions, base_rewards)):
        score = base_score
        
        # Anti-hacking: Check code STRUCTURE not just keywords
        # 1. Extract code blocks -- are there actual code blocks?
        code_blocks = re.findall(
            r'```(?:javascript|js|typescript|ts|html|css)?\n(.*?)```',
            completion, re.DOTALL
        )
        if code_blocks:
            code = '\n'.join(code_blocks)
            
            # 2. Import validation: check imports are actually used
            imports = re.findall(r'import\s+.*?from\s+[\'"](.*?)[\'"]', code)
            for imp in imports:
                module_name = imp.split('/')[-1].replace("'", "").replace('"', '')
                # Remove the import line itself, then check if module name appears
                code_without_import = re.sub(
                    rf'import\s+.*?from\s+[\'"].*?{re.escape(module_name)}.*?[\'"]',
                    '', code
                )
                if module_name.lower() not in code_without_import.lower():
                    score -= 1.0  # Unused import = potential keyword stuffing
            
            # 3. Variable definition checks: are variables defined before use?
            defined_vars = set(re.findall(r'(?:const|let|var)\s+(\w+)', code))
            used_vars = set(re.findall(r'(?<!(?:const|let|var)\s)(\b[a-z]\w+)\s*[.([\]]', code))
            # High ratio of used-but-undefined suggests copied/incoherent code
            if len(used_vars) > 0:
                undefined_ratio = len(used_vars - defined_vars) / len(used_vars)
                if undefined_ratio > 0.8:
                    score -= 2.0
            
            # 4. Diversity penalty: check for repeated patterns (reward gaming)
            lines = code.split('\n')
            non_empty_lines = [l for l in lines if l.strip()]
            if len(non_empty_lines) > 10:
                unique_lines = set(line.strip() for line in non_empty_lines)
                repetition_ratio = 1.0 - (len(unique_lines) / len(non_empty_lines))
                if repetition_ratio > 0.5:
                    score -= 3.0  # >50% repeated lines = reward gaming
        else:
            # No code blocks at all -- heavy penalty
            score -= 5.0
        
        enhanced_rewards.append(score)
    
    return enhanced_rewards

# Quick sanity test of the reward function
test_good = """Here's a Three.js particle system:
```javascript
import * as THREE from 'three';
import gsap from 'gsap';

const scene = new THREE.Scene();
const camera = new THREE.PerspectiveCamera(75, window.innerWidth / window.innerHeight, 0.1, 1000);
const renderer = new THREE.WebGLRenderer();
const particles = new THREE.BufferGeometry();
gsap.to(camera.position, { z: 5, duration: 2 });
```"""

test_slop = """<div class='hero-section bg-gradient-to-r from-purple-600 to-blue-500'>
  <h1 class='text-6xl font-bold text-white animate-bounce'>Welcome</h1>
</div>"""

print(f"Good code reward: {enhanced_creative_reward([test_good])[0]:.1f}")
print(f"Slop code reward: {enhanced_creative_reward([test_slop])[0]:.1f}")
print("Reward function working correctly" if enhanced_creative_reward([test_good])[0] > enhanced_creative_reward([test_slop])[0] else "WARNING: Reward function may need tuning")

## Cell 6: Load GRPO Prompt Dataset
GRPO uses prompt-only data. The model generates completions on its own, and the reward function scores them. This is different from SFT (which needs full responses) and SimPO (which needs chosen/rejected pairs).

In [ ]:
from datasets import load_dataset

# GRPO uses prompt-only data; model generates completions, reward scores them
prompts_dataset = load_dataset("json", data_files=GRPO_DATA_PATH)
print(f"GRPO prompts: {len(prompts_dataset['train'])}")
print(f"Sample: {prompts_dataset['train'][0]['prompt'][:200]}...")
print(f"\nDataset columns: {prompts_dataset['train'].column_names}")

## Cell 7: Configure and Run GRPO Training
Run GRPO with the enhanced reward function. Key settings:
- `num_generations=4`: generates 4 completions per prompt for group comparison
- `learning_rate=1e-5`: very low LR for RL stage (100x lower than SFT)
- `max_completion_length=4096`: allow full component-length generations

**Open Question 1 Fallback:** If Unsloth GRPO fails on Gemma 4 31B, the except block
provides instructions for falling back to TRL GRPOTrainer directly (load model with
AutoModelForCausalLM instead of Unsloth FastLanguageModel).

In [ ]:
from trl import GRPOTrainer, GRPOConfig
import os

os.environ["WANDB_PROJECT"] = WANDB_PROJECT

grpo_config = GRPOConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    learning_rate=LEARNING_RATE,
    bf16=BF16,
    max_completion_length=MAX_COMPLETION_LENGTH,
    num_generations=NUM_GENERATIONS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    report_to=REPORT_TO,
    run_name=WANDB_RUN_NAME,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[enhanced_creative_reward],
    train_dataset=prompts_dataset["train"],
    args=grpo_config,
)

print(f"Pre-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")

# NOTE: If Unsloth GRPO fails on Gemma 4 31B, fallback to TRL GRPOTrainer directly
# (Open Question 1 from research). Try Unsloth first.
try:
    trainer.train()
except Exception as e:
    print(f"\n{'='*60}")
    print(f"GRPO training error: {e}")
    print(f"{'='*60}")
    print("\nIf this is an Unsloth compatibility issue with Gemma 4 31B GRPO,")
    print("use the TRL-native fallback:")
    print("")
    print("  # Re-load model without Unsloth:")
    print("  from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig")
    print("  bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16)")
    print("  model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config)")
    print("  tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)")
    print("  # Then apply PEFT LoRA manually and re-create GRPOTrainer")
    print("")
    print("This approach uses more VRAM but avoids Unsloth-specific GRPO issues.")
    raise

print(f"\nPost-train VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f}GB")
print("GRPO training complete!")

## Cell 8: Monitor for Reward Hacking
**Pitfall 3:** If reward scores climb rapidly but code quality doesn't improve, the model is gaming the reward function.

Checks:
1. Compare early vs late reward scores from W&B logs
2. If increase > 10.0, flag potential reward hacking
3. Generate a sample and manually inspect + score it

**Action if reward hacking detected:** Do NOT proceed to KTO. Review generated samples, adjust the reward function, and retrain.

In [ ]:
# Pitfall 3: Check for reward hacking
# If reward scores climb rapidly but code quality doesn't improve, we have a problem
import wandb
api = wandb.Api()

try:
    run = api.run(f"{WANDB_PROJECT}/{WANDB_RUN_NAME}")
    history = run.history()
    
    if 'reward' in history.columns and len(history) > 20:
        early_reward = history['reward'].iloc[:10].mean()
        late_reward = history['reward'].iloc[-10:].mean()
        reward_increase = late_reward - early_reward
        
        if reward_increase > 10.0:
            print(f"WARNING: Reward increased by {reward_increase:.1f} -- possible reward hacking!")
            print("Check generated samples manually before proceeding to KTO.")
            print("Consider adjusting enhanced_creative_reward penalties.")
        else:
            print(f"Reward increase: {reward_increase:.1f} -- looks reasonable")
    else:
        print("Not enough training history to check for reward hacking.")
        print("Proceeding with manual sample check below.")
except Exception as e:
    print(f"Could not check W&B history: {e}")
    print("Proceeding with manual sample check below.")

# Generate a sample and check quality regardless
FastLanguageModel.for_inference(model)
test_prompt = "Create a WebGL shader that generates a noise-based terrain with GSAP-controlled camera dolly on scroll"
inputs = tokenizer.apply_chat_template(
    [{"role": "user", "content": test_prompt}],
    tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")
outputs = model.generate(input_ids=inputs, max_new_tokens=2048, temperature=0.7)
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"\n{'='*60}")
print(f"Test prompt: {test_prompt}")
print(f"{'='*60}")
print(response[:500])

reward = enhanced_creative_reward([response])
print(f"\nReward score: {reward[0]:.1f}")
print(f"Response length: {len(response)} chars")

## Cell 9: Merge Adapter and Save to Drive
Merge the GRPO-trained LoRA adapter into the base model weights. This merged model becomes the input for Stage 4 (KTO).

**Pipeline chain:** base -> SFT-merged -> SimPO-merged -> **GRPO-merged** -> KTO/final-merged

In [ ]:
model.save_pretrained_merged(MERGED_OUTPUT_DIR, tokenizer)
print(f"GRPO-merged model saved to {MERGED_OUTPUT_DIR}")

# Save to Drive for persistence across Colab sessions
!cp -r {MERGED_OUTPUT_DIR} /content/drive/MyDrive/cipher-models/
print("GRPO-merged model copied to Drive")
print(f"\nNext step: Run notebook 05_kto_training.ipynb")
print(f"KTO will load from: ./cipher-grpo-merged")